# 🎨 Krea-2 Turbo GGUF Studio Pro (Colab Edition)
### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

A high-performance deployment notebook for **Krea-2 Turbo GGUF** powered by **uv**, low-VRAM async streaming, and zero-config Cloudflare/Localtunnel deployment on Google Colab.

---

In [ ]:
#@title 🚀 Step 1: Initialize System, UV Acceleration & Cloudflare Engine
import os
import sys
import subprocess

def print_status(text, status_type="info"):
    icons = {"loading": "⏳", "info": "ℹ️", "success": "✅", "warning": "⚠️", "error": "❌"}
    icon = icons.get(status_type, "🔹")
    print(f"\n{icon} [Colab Engine] {text}")
    sys.stdout.flush()

print_status("Installing UV, aria2c, cloudflared, and system utilities...", "loading")
os.system("apt-get update -qq && apt-get install -y -qq aria2 nodejs npm curl")
os.system("pip install -q uv")
os.system("npm install -g localtunnel -q")

# Fetch official standalone Cloudflare Tunnel binary for instant HTTPS routing
if not os.path.exists("/usr/local/bin/cloudflared"):
    os.system("curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")

print_status("Cloning ComfyUI backend repository...", "loading")
if not os.path.exists("/content/ComfyUI"):
    os.system("git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI")

print_status("Resolving high-speed dependencies via UV package manager...", "loading")
os.system("uv pip install --system -q flask gguf torchsde websocket-client")
os.system("uv pip install --system -q -r /content/ComfyUI/requirements.txt")

print_status("Step 1 Complete: Environment configured with UV acceleration!", "success")

In [ ]:
#@title 📦 Step 2: Download Verified Models & Custom Nodes
#@markdown Select GGUF Quantization URL:
gguf_model_url = "https://huggingface.co/vantagewithai/Krea-2-Turbo-GGUF/resolve/main/krea2_turbo-Q5_K_M.gguf" #@param {type:"string"}

import os
import sys
import urllib.parse

def print_status(text, status_type="info"):
    icons = {"loading": "⏳", "info": "ℹ️", "success": "✅", "warning": "⚠️", "error": "❌"}
    icon = icons.get(status_type, "🔹")
    print(f"\n{icon} [Colab Engine] {text}")
    sys.stdout.flush()

custom_nodes_dir = "/content/ComfyUI/custom_nodes"
os.makedirs(custom_nodes_dir, exist_ok=True)

nodes = [
    ("ComfyUI-GGUF", "https://github.com/city96/ComfyUI-GGUF"),
    ("ComfyUI-Custom-Scripts", "https://github.com/pythongosssss/ComfyUI-Custom-Scripts")
]

for node_name, repo_url in nodes:
    node_path = os.path.join(custom_nodes_dir, node_name)
    if not os.path.exists(node_path):
        print_status(f"Setting up custom node: {node_name}...", "loading")
        os.system(f"git clone {repo_url} {node_path}")
        req_file = os.path.join(node_path, "requirements.txt")
        if os.path.exists(req_file):
            os.system(f"uv pip install --system -q -r {req_file}")
models_dir = "/content/ComfyUI/models"
unet_dir = os.path.join(models_dir, "unet")
clip_dir = os.path.join(models_dir, "clip")
text_enc_dir = os.path.join(models_dir, "text_encoders")
vae_dir = os.path.join(models_dir, "vae")

for d in [unet_dir, clip_dir, text_enc_dir, vae_dir]:
    os.makedirs(d, exist_ok=True)

parsed_url = urllib.parse.urlparse(gguf_model_url)
gguf_filename = os.path.basename(parsed_url.path)
if not gguf_filename.endswith('.gguf'):
    gguf_filename = "krea2_turbo-Q5_K_M.gguf"

# Direct Hugging Face verified download endpoints
downloads = [
    (unet_dir, gguf_filename, gguf_model_url),
    (clip_dir, "qwen3vl_4b_fp8_scaled.safetensors", "https://huggingface.co/Comfy-Org/Qwen3-VL/resolve/main/text_encoders/qwen3vl_4b_fp8_scaled.safetensors"),
    (vae_dir, "qwen_image_vae.safetensors", "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors")
]

for target_dir, filename, url in downloads:
    file_path = os.path.join(target_dir, filename)
    # Purge broken 0-byte downloads if any exist
    if os.path.exists(file_path) and os.path.getsize(file_path) < 1024 * 1024:
        os.remove(file_path)

    if not os.path.exists(file_path):
        print_status(f"Downloading {filename}...", "loading")
        os.system(f"aria2c --console-log-level=warn -c -x 16 -s 16 -k 1M '{url}' -d '{target_dir}' -o '{filename}'")
    else:
        print_status(f"Model cached: {filename}", "success")

# Symlink CLIP model into text_encoders folder for universal node compatibility
clip_src = os.path.join(clip_dir, "qwen3vl_4b_fp8_scaled.safetensors")
clip_dst = os.path.join(text_enc_dir, "qwen3vl_4b_fp8_scaled.safetensors")
if os.path.exists(clip_src) and not os.path.exists(clip_dst):
    os.symlink(clip_src, clip_dst)

print_status(f"Step 2 Complete: Weights verified and linked!", "success")

In [ ]:
#@title 🌐 Step 3: Launch Enhanced Krea-2 Turbo Web Studio
import os
import sys
import json
import time
import re
import socket
import random
import threading
import subprocess
import urllib.request
import urllib.parse
from flask import Flask, render_template_string, request, jsonify, send_from_directory
from IPython.display import display, HTML

def print_status(text, status_type="info"):
    icons = {"loading": "⏳", "info": "ℹ️", "success": "✅", "warning": "⚠️", "error": "❌"}
    icon = icons.get(status_type, "🔹")
    print(f"\n{icon} [Colab Engine] {text}")
    sys.stdout.flush()

def is_port_open(port=8188, host="127.0.0.1"):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(1.0)
    try:
        s.connect((host, port))
        s.close()
        return True
    except Exception:
        return False

server_process = None
if not is_port_open(8188):
    print_status("Starting ComfyUI backend with dynamic low-VRAM memory swapping...", "loading")
    comfy_dir = "/content/ComfyUI"
    if os.path.exists(comfy_dir):
        server_process = subprocess.Popen(
            [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--lowvram", "--dont-upcast-attention"],
            cwd=comfy_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1
        )
        for _ in range(45):
            if is_port_open(8188):
                print_status("ComfyUI engine confirmed online on port 8188!", "success")
                break
            time.sleep(1)
        else:
            print_status("ComfyUI engine initialization is taking longer than expected...", "warning")
    else:
        print_status("Directory /content/ComfyUI missing. Run Step 1 first.", "error")
else:
    print_status("ComfyUI engine already active on port 8188.", "success")

app = Flask(__name__)
task_store = {}

HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en" class="dark">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Krea-2 Turbo GGUF Studio Pro</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link href="https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap" rel="stylesheet">
    <style>
        * { font-family: 'Plus Jakarta Sans', system-ui, sans-serif; }
        body { background: radial-gradient(circle at 50% 0%, #151c2e 0%, #080b11 100%); min-height: 100vh; color: #e2e8f0; }
        .glass { background: rgba(15, 23, 42, 0.65); backdrop-filter: blur(16px); border: 1px solid rgba(255, 255, 255, 0.08); }
        .glow-btn { background: linear-gradient(135deg, #3b82f6 0%, #2563eb 50%, #1d4ed8 100%); transition: all 0.25s ease; }
        .glow-btn:hover:not(:disabled) { box-shadow: 0 0 25px rgba(59, 130, 246, 0.6); transform: translateY(-2px); }
        .custom-scroll::-webkit-scrollbar { width: 6px; height: 6px; }
        .custom-scroll::-webkit-scrollbar-track { background: rgba(0, 0, 0, 0.2); border-radius: 4px; }
        .custom-scroll::-webkit-scrollbar-thumb { background: rgba(255, 255, 255, 0.15); border-radius: 4px; }
        .custom-scroll::-webkit-scrollbar-thumb:hover { background: rgba(255, 255, 255, 0.3); }
    </style>
</head>
<body class="p-4 md:p-8 flex flex-col items-center custom-scroll">
    <div class="w-full max-w-6xl space-y-6">
        <!-- Top Bar -->
        <header class="glass rounded-2xl p-5 md:p-6 flex items-center justify-between shadow-2xl shadow-blue-950/20">
            <div class="flex items-center space-x-3.5">
                <div class="w-11 h-11 rounded-xl bg-gradient-to-br from-blue-500 to-indigo-600 flex items-center justify-center text-xl shadow-lg shadow-blue-500/25">⚡</div>
                <div>
                    <h1 class="text-xl md:text-2xl font-extrabold tracking-tight text-transparent bg-clip-text bg-gradient-to-r from-white via-slate-100 to-blue-300">Krea-2 Turbo Studio Pro</h1>
                    <p class="text-xs text-slate-400 font-medium">Brought to you by <a href="https://youtube.com/@AIWithChucky" target="_blank" class="text-red-400 hover:text-red-300 font-bold underline transition">AI With Chucky</a></p>
                </div>
            </div>
            <div class="flex items-center space-x-3">
                <span class="inline-flex items-center px-3 py-1 rounded-full text-xs font-semibold bg-emerald-500/10 text-emerald-400 border border-emerald-500/20">
                    <span class="w-1.5 h-1.5 rounded-full bg-emerald-400 mr-2 animate-pulse"></span>GPU Active
                </span>
            </div>
        </header>

        <!-- Main Studio Grid -->
        <div class="grid grid-cols-1 lg:grid-cols-12 gap-6">
            <!-- Left Config Panel -->
            <div class="lg:col-span-5 glass rounded-2xl p-6 space-y-5 flex flex-col justify-between shadow-xl">
                <div class="space-y-4">
                    <!-- Prompt -->
                    <div>
                        <div class="flex justify-between items-center mb-1.5">
                            <label class="text-xs font-bold uppercase tracking-wider text-slate-300">Prompt</label>
                            <button onclick="randomizePrompt()" class="text-xs text-blue-400 hover:text-blue-300 transition">🎲 Example</button>
                        </div>
                        <textarea id="promptText" rows="4" class="w-full bg-slate-900/90 border border-slate-700/80 rounded-xl p-3 text-sm focus:outline-none focus:ring-2 focus:ring-blue-500/50 focus:border-blue-500 transition text-slate-100 placeholder-slate-500 resize-none" placeholder="Describe what you want to render...">A high-resolution, surreal digital illustration showing a human hand holding a glowing martini glass, surrounded by intricate neon ink doodles, clean minimal dark background.</textarea>
                    </div>

                    <!-- Negative Prompt -->
                    <div>
                        <label class="block text-xs font-bold uppercase tracking-wider text-slate-400 mb-1.5">Negative Prompt (Optional)</label>
                        <input type="text" id="negativePrompt" placeholder="blurry, low quality, distorted, artifacts" class="w-full bg-slate-900/90 border border-slate-700/80 rounded-xl px-3 py-2 text-xs focus:outline-none focus:ring-2 focus:ring-blue-500/50 text-slate-200">
                    </div>

                    <!-- Dimension Presets -->
                    <div>
                        <label class="block text-xs font-bold uppercase tracking-wider text-slate-300 mb-1.5">Aspect Ratio & Resolution</label>
                        <select id="resolution" class="w-full bg-slate-900/90 border border-slate-700/80 rounded-xl p-2.5 text-xs font-medium focus:outline-none focus:ring-2 focus:ring-blue-500/50 text-slate-200">
                            <option value="1024x1024" selected>1:1 Square (1024 × 1024)</option>
                            <option value="832x1216">9:16 Portrait (832 × 1216)</option>
                            <option value="1216x832">16:9 Landscape (1216 × 832)</option>
                            <option value="768x1344">9:21 Tall (768 × 1344)</option>
                            <option value="1344x768">21:9 Ultra-Wide (1344 × 768)</option>
                        </select>
                    </div>

                    <!-- Sliders & Inputs -->
                    <div class="grid grid-cols-2 gap-3.5 pt-1">
                        <div>
                            <div class="flex justify-between text-xs font-bold uppercase tracking-wider text-slate-400 mb-1">
                                <span>Steps</span>
                                <span id="stepsVal" class="text-blue-400">8</span>
                            </div>
                            <input type="range" id="steps" min="4" max="20" value="8" oninput="document.getElementById('stepsVal').innerText=this.value" class="w-full accent-blue-500 h-1.5 bg-slate-800 rounded-lg cursor-pointer">
                        </div>
                        <div>
                            <div class="flex justify-between text-xs font-bold uppercase tracking-wider text-slate-400 mb-1">
                                <span>CFG Scale</span>
                                <span id="cfgVal" class="text-blue-400">1.0</span>
                            </div>
                            <input type="range" id="cfg" min="1.0" max="3.5" step="0.1" value="1.0" oninput="document.getElementById('cfgVal').innerText=this.value" class="w-full accent-blue-500 h-1.5 bg-slate-800 rounded-lg cursor-pointer">
                        </div>
                    </div>

                    <!-- Seed & Sampler -->
                    <div class="grid grid-cols-2 gap-3.5">
                        <div>
                            <label class="block text-xs font-bold uppercase tracking-wider text-slate-400 mb-1">Seed</label>
                            <div class="relative flex items-center">
                                <input type="number" id="seed" value="-1" class="w-full bg-slate-900/90 border border-slate-700/80 rounded-xl px-2.5 py-2 text-xs focus:outline-none focus:ring-2 focus:ring-blue-500/50 text-slate-200">
                                <button onclick="document.getElementById('seed').value = -1" title="Randomize" class="absolute right-2 text-slate-400 hover:text-white text-xs">🎲</button>
                            </div>
                        </div>
                        <div>
                            <label class="block text-xs font-bold uppercase tracking-wider text-slate-400 mb-1">Sampler</label>
                            <select id="sampler" class="w-full bg-slate-900/90 border border-slate-700/80 rounded-xl px-2.5 py-2 text-xs focus:outline-none focus:ring-2 focus:ring-blue-500/50 text-slate-200">
                                <option value="euler" selected>euler / simple</option>
                                <option value="dpmpp_2m">dpmpp_2m / karras</option>
                                <option value="ddim">ddim / normal</option>
                            </select>
                        </div>
                    </div>
                </div>

                <!-- Generate Button -->
                <button onclick="generateImage()" id="generateBtn" class="w-full glow-btn text-white font-bold py-3.5 rounded-xl flex items-center justify-center space-x-2 mt-4 cursor-pointer shadow-lg">
                    <span id="btnIcon">✨</span>
                    <span id="btnText">Generate Artwork</span>
                </button>
            </div>

            <!-- Right Canvas / Output Panel -->
            <div class="lg:col-span-7 glass rounded-2xl p-6 flex flex-col justify-between min-h-[480px] shadow-xl relative overflow-hidden">
                <!-- Render Stage -->
                <div class="flex-1 flex flex-col items-center justify-center relative min-h-[360px]">
                    <!-- Placeholder -->
                    <div id="placeholderState" class="text-center space-y-3 p-8 select-none">
                        <div class="w-20 h-20 rounded-2xl bg-slate-800/80 border border-slate-700/80 flex items-center justify-center mx-auto text-3xl shadow-inner">🎨</div>
                        <h3 class="text-base font-semibold text-slate-300">Your Canvas is Empty</h3>
                        <p class="text-slate-500 text-xs max-w-sm">Enter a concept and click Generate to produce crystal clear high-resolution outputs.</p>
                    </div>

                    <!-- Active Progress State -->
                    <div id="loadingState" class="hidden flex flex-col items-center space-y-4 w-full max-w-md px-6 text-center">
                        <div class="relative flex items-center justify-center">
                            <div class="w-16 h-16 border-4 border-blue-500/20 border-t-blue-500 rounded-full animate-spin"></div>
                            <span id="timerLabel" class="absolute text-xs font-bold text-blue-300">0.0s</span>
                        </div>
                        <div class="w-full space-y-2">
                            <div class="flex justify-between text-xs text-slate-400 font-medium">
                                <span id="statusText">Submitting to Colab GPU...</span>
                                <span id="progressPercent">0%</span>
                            </div>
                            <div class="w-full bg-slate-800 rounded-full h-2 overflow-hidden">
                                <div id="progressBar" class="bg-gradient-to-r from-blue-500 to-indigo-500 h-2 rounded-full transition-all duration-300" style="width: 5%"></div>
                            </div>
                        </div>
                    </div>

                    <!-- Final Image -->
                    <div id="imageContainer" class="hidden relative group max-h-[460px]">
                        <img id="resultImage" class="max-h-[440px] w-auto rounded-xl shadow-2xl object-contain border border-slate-700/80 cursor-pointer" alt="Rendered Image" onclick="openLightbox(this.src)">
                        <div class="absolute bottom-3 right-3 flex items-center space-x-2 opacity-0 group-hover:opacity-100 transition-opacity duration-200 bg-slate-950/80 backdrop-blur-md p-1.5 rounded-xl border border-white/10">
                            <button onclick="downloadCurrentImage()" class="p-2 hover:bg-blue-600 rounded-lg text-xs font-semibold text-white flex items-center space-x-1 transition">
                                <span>⬇️</span><span>Save</span>
                            </button>
                            <button onclick="openLightbox(document.getElementById('resultImage').src)" class="p-2 hover:bg-slate-700 rounded-lg text-xs text-white transition">
                                <span>🔍</span>
                            </button>
                        </div>
                    </div>
                </div>

                <!-- Session History Strip -->
                <div class="pt-4 border-t border-slate-800/80 mt-2">
                    <div class="flex justify-between items-center mb-2">
                        <span class="text-[11px] font-bold uppercase tracking-wider text-slate-400">Session Gallery</span>
                        <span id="historyCount" class="text-[11px] text-slate-500">0 items</span>
                    </div>
                    <div id="historyStrip" class="flex space-x-3 overflow-x-auto pb-1 custom-scroll min-h-[58px] items-center text-xs text-slate-600">
                        <span>Generated pictures will collect here.</span>
                    </div>
                </div>
            </div>
        </div>
    </div>

    <!-- Lightbox Modal -->
    <div id="lightbox" class="fixed inset-0 bg-black/90 backdrop-blur-md z-50 hidden flex items-center justify-center p-4" onclick="closeLightbox()">
        <img id="lightboxImg" class="max-w-[92vw] max-h-[92vh] object-contain rounded-xl border border-white/10 shadow-2xl" src="" alt="Zoomed View">
    </div>

    <script>
        let pollTimer = null;
        let elapsedTimer = null;
        let startTime = 0;
        let sessionHistory = [];

        const samplePrompts = [
            "A high-resolution, surreal digital illustration showing a human hand holding a glowing martini glass, surrounded by intricate neon ink doodles, clean minimal dark background.",
            "Cinematic 3D render of an astronaut meditating on a glass monolith floating in nebula space, iridescent lighting, 8k masterpiece.",
            "Editorial studio portrait of a futuristic cyberpunk model with delicate metallic gold floral cyberware, soft rim lighting, hyperdetailed.",
            "Macro photograph of a fantasy biome inside a water droplet on an emerald leaf, vivid bioluminescent colors, depth of field."
        ];

        function randomizePrompt() {
            const p = samplePrompts[Math.floor(Math.random() * samplePrompts.length)];
            document.getElementById('promptText').value = p;
        }

        async function generateImage() {
            const btn = document.getElementById('generateBtn');
            const btnText = document.getElementById('btnText');
            const placeholder = document.getElementById('placeholderState');
            const loading = document.getElementById('loadingState');
            const imageContainer = document.getElementById('imageContainer');
            const statusText = document.getElementById('statusText');
            const progressBar = document.getElementById('progressBar');
            const progressPercent = document.getElementById('progressPercent');
            const timerLabel = document.getElementById('timerLabel');
            const res = document.getElementById('resolution').value.split('x');

            btn.disabled = true;
            btn.classList.add('opacity-50', 'cursor-not-allowed');
            btnText.innerText = 'Synthesizing...';

            placeholder.classList.add('hidden');
            imageContainer.classList.add('hidden');
            loading.classList.remove('hidden');
            
            statusText.innerText = 'Queuing GPU task...';
            progressBar.style.width = '15%';
            progressPercent.innerText = '15%';

            startTime = Date.now();
            if (elapsedTimer) clearInterval(elapsedTimer);
            elapsedTimer = setInterval(() => {
                const secs = ((Date.now() - startTime) / 1000).toFixed(1);
                timerLabel.innerText = `${secs}s`;
            }, 100);

            const payload = {
                prompt: document.getElementById('promptText').value,
                negative: document.getElementById('negativePrompt').value,
                steps: parseInt(document.getElementById('steps').value),
                cfg: parseFloat(document.getElementById('cfg').value),
                seed: parseInt(document.getElementById('seed').value),
                sampler: document.getElementById('sampler').value,
                width: parseInt(res[0]),
                height: parseInt(res[1])
            };

            try {
                const response = await fetch('/api/generate', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify(payload)
                });
                
                const data = await response.json();
                if (data.success) {
                    pollTaskStatus(data.prompt_id);
                } else {
                    alert('Generation Error: ' + data.error);
                    resetUI();
                }
            } catch (err) {
                alert('Server connection interrupted. Please verify backend cell.');
                resetUI();
            }
        }

        function pollTaskStatus(promptId) {
            const statusText = document.getElementById('statusText');
            const progressBar = document.getElementById('progressBar');
            const progressPercent = document.getElementById('progressPercent');
            let estimatedProgress = 25;

            if (pollTimer) clearInterval(pollTimer);

            pollTimer = setInterval(async () => {
                try {
                    estimatedProgress = Math.min(estimatedProgress + 15, 90);
                    progressBar.style.width = estimatedProgress + '%';
                    progressPercent.innerText = estimatedProgress + '%';
                    statusText.innerText = 'Denoising latents (Krea-2 Turbo)...';

                    const res = await fetch('/api/task/' + promptId);
                    const task = await res.json();

                    if (task.status === 'completed') {
                        clearInterval(pollTimer);
                        clearInterval(elapsedTimer);
                        
                        progressBar.style.width = '100%';
                        progressPercent.innerText = '100%';
                        statusText.innerText = 'Decoding VAE...';
                        
                        setTimeout(() => {
                            const resultImg = document.getElementById('resultImage');
                            const loading = document.getElementById('loadingState');
                            const imageContainer = document.getElementById('imageContainer');
                            
                            const imgUrl = task.image_url + '?t=' + Date.now();
                            resultImg.src = imgUrl;
                            imageContainer.classList.remove('hidden');
                            loading.classList.add('hidden');
                            
                            addToHistory(imgUrl);
                            resetBtn();
                        }, 400);
                    } else if (task.status === 'error') {
                        clearInterval(pollTimer);
                        clearInterval(elapsedTimer);
                        alert('Pipeline Error: ' + task.error);
                        resetUI();
                    }
                } catch (e) {
                    statusText.innerText = 'Polling GPU state...';
                }
            }, 1200);
        }

        function addToHistory(url) {
            sessionHistory.unshift(url);
            const strip = document.getElementById('historyStrip');
            document.getElementById('historyCount').innerText = `${sessionHistory.length} items`;
            
            strip.innerHTML = '';
            sessionHistory.forEach((u) => {
                const thumb = document.createElement('img');
                thumb.src = u;
                thumb.className = 'w-14 h-14 rounded-lg object-cover border border-slate-700 hover:border-blue-500 cursor-pointer flex-shrink-0 transition hover:scale-105';
                thumb.onclick = () => {
                    document.getElementById('resultImage').src = u;
                    document.getElementById('placeholderState').classList.add('hidden');
                    document.getElementById('imageContainer').classList.remove('hidden');
                };
                strip.appendChild(thumb);
            });
        }

        function downloadCurrentImage() {
            const img = document.getElementById('resultImage');
            if (!img.src) return;
            const a = document.createElement('a');
            a.href = img.src;
            a.download = `krea2_turbo_${Date.now()}.png`;
            a.click();
        }

        function openLightbox(src) {
            const lb = document.getElementById('lightbox');
            document.getElementById('lightboxImg').src = src;
            lb.classList.remove('hidden');
        }

        function closeLightbox() {
            document.getElementById('lightbox').classList.add('hidden');
        }

        function resetBtn() {
            const btn = document.getElementById('generateBtn');
            const btnText = document.getElementById('btnText');
            btn.disabled = false;
            btn.classList.remove('opacity-50', 'cursor-not-allowed');
            btnText.innerText = 'Generate Artwork';
        }

        function resetUI() {
            if (elapsedTimer) clearInterval(elapsedTimer);
            document.getElementById('placeholderState').classList.remove('hidden');
            document.getElementById('loadingState').classList.add('hidden');
            resetBtn();
        }
    </script>
</body>
</html>
"""

@app.route('/')
def index():
    return render_template_string(HTML_TEMPLATE)

@app.route('/output/<filename>')
def serve_output(filename):
    return send_from_directory('/content/ComfyUI/output', filename)

@app.route('/api/task/<prompt_id>', methods=['GET'])
def get_task_status(prompt_id):
    task = task_store.get(prompt_id)
    if not task:
        return jsonify({"status": "unknown", "error": "Task ID not found"}), 404
    return jsonify(task)

def monitor_gpu_task(prompt_id):
    while True:
        try:
            h_req = urllib.request.Request(f"http://127.0.0.1:8188/history/{prompt_id}")
            h_res = json.loads(urllib.request.urlopen(h_req, timeout=5).read())
            if prompt_id in h_res:
                outputs = h_res[prompt_id].get('outputs', {})
                for node_id, node_out in outputs.items():
                    if 'images' in node_out and len(node_out['images']) > 0:
                        filename = node_out['images'][0]['filename']
                        task_store[prompt_id] = {
                            "status": "completed",
                            "image_url": f"/output/{filename}"
                        }
                        return
                task_store[prompt_id] = {"status": "completed", "image_url": ""}
                return
        except Exception:
            pass
        time.sleep(1.0)

@app.route('/api/generate', methods=['POST'])
def generate():
    if not is_port_open(8188):
        return jsonify({"success": False, "error": "ComfyUI engine on port 8188 is offline."})

    data = request.json or {}
    user_prompt = data.get('prompt', 'A beautiful landscape')
    neg_prompt = data.get('negative', '').strip()
    steps = data.get('steps', 8)
    cfg = data.get('cfg', 1.0)
    seed = data.get('seed', -1)
    sampler_name = data.get('sampler', 'euler')
    width = data.get('width', 1024)
    height = data.get('height', 1024)

    scheduler_map = {'euler': 'simple', 'dpmpp_2m': 'karras', 'ddim': 'normal'}
    scheduler = scheduler_map.get(sampler_name, 'simple')

    if seed == -1:
        seed = random.randint(1, 10**14)

    unet_dir = '/content/ComfyUI/models/unet'
    unet_files = [f for f in os.listdir(unet_dir) if f.endswith('.gguf')]
    selected_unet = unet_files[0] if unet_files else "krea2_turbo-Q5_K_M.gguf"

    # Dynamic negative conditioning routing
    workflow_payload = {
      "1": {"inputs": {"unet_name": selected_unet}, "class_type": "UnetLoaderGGUF"},
      "2": {"inputs": {"clip_name": "qwen3vl_4b_fp8_scaled.safetensors", "type": "krea2", "device": "default"}, "class_type": "CLIPLoader"},
      "3": {"inputs": {"vae_name": "qwen_image_vae.safetensors"}, "class_type": "VAELoader"},
      "4": {"inputs": {"text": user_prompt, "clip": ["2", 0]}, "class_type": "CLIPTextEncode"},
      "6": {"inputs": {"width": width, "height": height, "batch_size": 1}, "class_type": "EmptyLatentImage"},
      "7": {
        "inputs": {
          "seed": seed,
          "steps": steps,
          "cfg": cfg,
          "sampler_name": sampler_name,
          "scheduler": scheduler,
          "denoise": 1.0,
          "model": ["1", 0],
          "positive": ["4", 0],
          "negative": ["5", 0],
          "latent_image": ["6", 0]
        },
        "class_type": "KSampler"
      },
      "8": {"inputs": {"samples": ["7", 0], "vae": ["3", 0]}, "class_type": "VAEDecode"},
      "9": {"inputs": {"filename_prefix": "Krea2_Studio", "images": ["8", 0]}, "class_type": "SaveImage"}
    }

    if neg_prompt:
        workflow_payload["5"] = {"inputs": {"text": neg_prompt, "clip": ["2", 0]}, "class_type": "CLIPTextEncode"}
    else:
        workflow_payload["5"] = {"inputs": {"conditioning": ["4", 0]}, "class_type": "ConditioningZeroOut"}

    try:
        p = {"prompt": workflow_payload}
        req = urllib.request.Request("http://127.0.0.1:8188/prompt", data=json.dumps(p).encode('utf-8'), headers={'Content-Type': 'application/json'})
        res = json.loads(urllib.request.urlopen(req, timeout=10).read())
        pid = res['prompt_id']

        task_store[pid] = {"status": "processing"}
        worker_thread = threading.Thread(target=monitor_gpu_task, args=(pid,), daemon=True)
        worker_thread.start()

        return jsonify({"success": True, "prompt_id": pid})
    except Exception as e:
        return jsonify({"success": False, "error": str(e)})

def run_flask():
    app.run(port=5000, host="127.0.0.1", use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)

# 1. Launch Cloudflare Tunnel (Zero password, direct link)
cf_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
cf_url = ""
for _ in range(25):
    line = cf_proc.stdout.readline()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        cf_url = m.group(0)
        break
    time.sleep(0.3)

# 2. Launch Localtunnel as secondary tunnel
lt_proc = subprocess.Popen(["lt", "--port", "5000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
lt_url = ""
for _ in range(15):
    line = lt_proc.stdout.readline()
    if "url is:" in line:
        lt_url = line.split("url is:")[-1].strip()
        break
    time.sleep(0.3)

try:
    public_ip = subprocess.check_output(["curl", "-s", "ipv4.icanhazip.com"]).decode().strip()
except Exception:
    public_ip = "N/A"

display(HTML(f"""
<div style="padding: 20px; border-radius: 16px; background: linear-gradient(135deg, #0b0f19 0%, #111827 100%); border: 1px solid #3b82f6; color: #f3f4f6; font-family: system-ui, sans-serif; box-shadow: 0 10px 30px rgba(0,0,0,0.5);">
    <div style="display: flex; align-items: center; margin-bottom: 12px;">
        <span style="font-size: 24px; margin-right: 10px;">🎨</span>
        <h2 style="margin: 0; color: #60a5fa; font-size: 20px; font-weight: 700;">Krea-2 Turbo Studio Pro Online</h2>
    </div>
    <p style="color: #94a3b8; font-size: 13px; margin-top: 0; margin-bottom: 16px;">Choose any of the live access links below to open your studio:</p>
    
    <div style="display: flex; flex-direction: column; gap: 10px;">
        <div style="background: rgba(59, 130, 246, 0.1); border: 1px solid rgba(59, 130, 246, 0.3); padding: 12px 16px; border-radius: 10px;">
            <span style="color: #60a5fa; font-weight: 600; font-size: 12px; text-transform: uppercase;">⚡ Primary Link (Direct • No Password):</span><br/>
            <a href="{cf_url}" target="_blank" style="color: #ffffff; font-weight: 700; font-size: 15px; text-decoration: underline;">{cf_url if cf_url else 'Building tunnel...'}</a>
        </div>
        <div style="background: rgba(255, 255, 255, 0.05); border: 1px solid rgba(255, 255, 255, 0.1); padding: 12px 16px; border-radius: 10px;">
            <span style="color: #a78bfa; font-weight: 600; font-size: 12px; text-transform: uppercase;">🌐 Backup Link (Localtunnel):</span><br/>
            <a href="{lt_url}" target="_blank" style="color: #e2e8f0; font-weight: 600; font-size: 14px; text-decoration: underline;">{lt_url if lt_url else 'Building backup tunnel...'}</a>
            <span style="color: #64748b; font-size: 12px; margin-left: 10px;">(Password: <strong style="color:#34d399;">{public_ip}</strong>)</span>
        </div>
    </div>
</div>
"""))

print_status("Streaming live GPU & ComfyUI logs below:", "info")
try:
    while True:
        if server_process and server_process.stdout:
            line = server_process.stdout.readline()
            if line:
                print(f"[ComfyUI Engine] {line.strip()}")
        time.sleep(0.05)
except KeyboardInterrupt:
    print_status("Studio loop terminated by user.", "info")